# AWS Cloud Analytics Pipeline: NYC Taxi Trip Data

Real pipeline: `download_data.py` pulled the latest month (May 2026) of NYC TLC yellow taxi trip data from the public CloudFront endpoint and uploaded it to a personal S3 bucket. An AWS Glue crawler cataloged the schema, and Amazon Athena was used to query it. See `README.md` for the full architecture writeup and findings summary.

In [ ]:
import boto3
import pandas as pd
import matplotlib.pyplot as plt

PROFILE_NAME = "preethi-portfolio"
REGION = "us-east-2"

session = boto3.Session(profile_name=PROFILE_NAME, region_name=REGION)
print("Authenticated as:", session.client("sts").get_caller_identity()["Arn"])

## Step 1: Glue Data Catalog schema

The Glue crawler (`nyc-taxi-portfolio-crawler`) inferred this schema directly from the Parquet file into `nyc_taxi_portfolio_db.trip_data`:

```
vendorid int
tpep_pickup_datetime timestamp
tpep_dropoff_datetime timestamp
passenger_count bigint
trip_distance double
ratecodeid bigint
store_and_fwd_flag string
pulocationid int
dolocationid int
payment_type bigint
fare_amount double
extra double
mta_tax double
tip_amount double
tolls_amount double
improvement_surcharge double
total_amount double
congestion_surcharge double
airport_fee double
cbd_congestion_fee double
```

## Step 2: Query via Athena

Two real queries were run against the cataloged table.

In [ ]:
import time
import io

athena = session.client("athena")
s3 = session.client("s3")

DATABASE = "nyc_taxi_portfolio_db"
TABLE = "trip_data"
BUCKET = "preethi-portfolio-nyc-taxi-099771438185"
ATHENA_RESULTS_BUCKET = f"s3://{BUCKET}/athena-results/"


def run_athena_query(sql):
    exec_id = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DATABASE},
        ResultConfiguration={"OutputLocation": ATHENA_RESULTS_BUCKET},
    )["QueryExecutionId"]

    while True:
        status = athena.get_query_execution(QueryExecutionId=exec_id)["QueryExecution"]["Status"]["State"]
        if status in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(2)

    if status != "SUCCEEDED":
        raise RuntimeError(f"Athena query {status}")

    obj = s3.get_object(Bucket=BUCKET, Key=f"athena-results/{exec_id}.csv")
    return pd.read_csv(io.BytesIO(obj["Body"].read()))


hourly_sql = f"""
SELECT
    hour(tpep_pickup_datetime) AS pickup_hour,
    COUNT(*) AS trip_count,
    AVG(date_diff('second', tpep_pickup_datetime, tpep_dropoff_datetime)) AS avg_trip_duration_sec,
    AVG(trip_distance) AS avg_trip_distance_miles,
    AVG(fare_amount) AS avg_fare_amount
FROM {TABLE}
WHERE tpep_dropoff_datetime > tpep_pickup_datetime
GROUP BY hour(tpep_pickup_datetime)
ORDER BY pickup_hour
"""

hourly_df = run_athena_query(hourly_sql)
hourly_df

## Step 3: Visualize

Trip volume peaks in the early evening (5-6 PM), and average trip duration peaks mid-afternoon (2-3 PM) despite shorter average distances -- consistent with mid-afternoon traffic congestion rather than longer trips. See `README.md` for the full findings writeup.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(hourly_df["pickup_hour"], hourly_df["trip_count"], color="#1f77b4")
axes[0].set_title("NYC Yellow Taxi Trips by Pickup Hour (May 2026)")
axes[0].set_xlabel("Hour of Day")
axes[0].set_ylabel("Trip Count")

axes[1].plot(hourly_df["pickup_hour"], hourly_df["avg_trip_duration_sec"] / 60, marker="o", color="#d62728")
axes[1].set_title("Average Trip Duration by Pickup Hour")
axes[1].set_xlabel("Hour of Day")
axes[1].set_ylabel("Avg Duration (minutes)")

plt.tight_layout()
plt.savefig("nyc_taxi_hourly_trends.png", dpi=120)
plt.show()